# Carga de datos

En un proceso productivo los datos del modelo estarían en el DWH o Datalake de la empresa, como resultado de un proceso que materializa la información en una tabla. Para este proyecto se usa un dataset no productivo de ejemplo en formato `.csv` (`Base_de_datos.csv`), que contiene información de créditos otorgados y si el cliente pagó a tiempo o no.

En este notebook se carga la base, se verifica que tenga la estructura esperada y se hace una revisión inicial antes del análisis exploratorio (`eda_analisis.ipynb`).

In [1]:
import json
from pathlib import Path

import pandas as pd

### Ruta del archivo y parámetros

La ruta y el separador se leen desde `config.json`, el mismo archivo de configuración que usan los scripts del pipeline, para que todo el proyecto lea los datos de la misma forma.

In [2]:
# El notebook puede ejecutarse desde la carpeta mlops_pipeline o desde la raíz
RAIZ = Path.cwd().parent if Path.cwd().name == 'mlops_pipeline' else Path.cwd()

with open(RAIZ / 'config.json', encoding='utf-8') as archivo:
    config = json.load(archivo)

ruta_datos = RAIZ / config['datos']['ruta']
separador = config['datos']['separador']
variable_objetivo = config['datos']['variable_objetivo']

print('Archivo:', config['datos']['ruta'])
print('Existe:', ruta_datos.exists())
print('Separador:', repr(separador))

Archivo: Base_de_datos.csv
Existe: True
Separador: ';'


### Carga de la base

In [3]:
df = pd.read_csv(ruta_datos, sep=separador)

print(f'Registros: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')

Registros: 10,763
Columnas: 23


In [4]:
df.head()

,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,4,7/01/2025 14:40,1852560,12,32,Empleado,3500000,1000000,128650,"95,227787",...,0.0,0.0,NaN,NaN,0,0,0,916148.0,Creciente,1
1,4,9/01/2025 11:18,3181080,6,34,Empleado,5000000,2000000,441817,"95,227787",...,0.0,0.0,NaN,NaN,0,0,0,4473774.0,Creciente,1
2,9,26/12/2024 18:52,670200,5,34,Independiente,5000000,2000000,108632,"95,227787",...,0.0,274561.0,274561.0,NaN,2,0,1,NaN,NaN,1
3,9,4/12/2024 14:20,506807,2,25,Independiente,3000000,900000,199684,"95,227787",...,0.0,27564.0,27564.0,NaN,1,0,6,NaN,NaN,1
4,4,30/04/2025 18:41,999780,10,26,Empleado,2000000,600000,92509,"95,227787",...,0.0,0.0,NaN,NaN,0,0,0,61000.0,Creciente,1


### Validación de la estructura

Se verifica que la base tenga exactamente las columnas que espera el pipeline. Si llega una base con columnas faltantes o con nombres diferentes, el proceso debe detenerse antes de continuar.

In [5]:
columnas_esperadas = [
    'tipo_credito', 'fecha_prestamo', 'capital_prestado', 'plazo_meses',
    'edad_cliente', 'tipo_laboral', 'salario_cliente', 'total_otros_prestamos',
    'cuota_pactada', 'puntaje', 'puntaje_datacredito', 'cant_creditosvigentes',
    'huella_consulta', 'saldo_mora', 'saldo_total', 'saldo_principal',
    'saldo_mora_codeudor', 'creditos_sectorFinanciero', 'creditos_sectorCooperativo',
    'creditos_sectorReal', 'promedio_ingresos_datacredito', 'tendencia_ingresos',
    'Pago_atiempo'
]

faltantes = set(columnas_esperadas) - set(df.columns)
adicionales = set(df.columns) - set(columnas_esperadas)

print('Columnas faltantes:', faltantes if faltantes else 'ninguna')
print('Columnas adicionales:', adicionales if adicionales else 'ninguna')

assert not faltantes, f'Faltan columnas en la base: {faltantes}'
assert df.shape[0] > 0, 'La base está vacía'

Columnas faltantes: ninguna
Columnas adicionales: ninguna


In [6]:
df.dtypes

tipo_credito                       int64
fecha_prestamo                       str
capital_prestado                   int64
plazo_meses                        int64
edad_cliente                       int64
tipo_laboral                         str
salario_cliente                    int64
total_otros_prestamos              int64
cuota_pactada                      int64
puntaje                              str
puntaje_datacredito              float64
cant_creditosvigentes              int64
huella_consulta                    int64
saldo_mora                       float64
saldo_total                      float64
saldo_principal                  float64
saldo_mora_codeudor              float64
creditos_sectorFinanciero          int64
creditos_sectorCooperativo         int64
creditos_sectorReal                int64
promedio_ingresos_datacredito    float64
tendencia_ingresos                   str
Pago_atiempo                       int64
dtype: object

Al cargar la base se observa que `fecha_prestamo` y `puntaje` quedan como texto. En el caso de `puntaje` esto ocurre porque usa coma como separador decimal (por ejemplo `95,227787`), y `fecha_prestamo` viene en formato día/mes/año con hora. Estas conversiones se hacen en el notebook de EDA y en `ft_eng.py`; en la carga se conservan los datos tal como vienen de la fuente.

### Revisión inicial de calidad

In [7]:
resumen_carga = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isnull().sum(),
    'porcentaje_nulos': (df.isnull().mean() * 100).round(2),
    'valores_unicos': df.nunique()
})

resumen_carga

,tipo,nulos,porcentaje_nulos,valores_unicos
tipo_credito,int64,0,0.00,6
fecha_prestamo,str,0,0.00,10444
capital_prestado,int64,0,0.00,7306
plazo_meses,int64,0,0.00,18
edad_cliente,int64,0,0.00,54
tipo_laboral,str,0,0.00,2
salario_cliente,int64,0,0.00,1385
total_otros_prestamos,int64,0,0.00,1538
cuota_pactada,int64,0,0.00,9836
puntaje,str,0,0.00,248


In [8]:
print('Filas duplicadas:', df.duplicated().sum())
print('Valores de la variable objetivo:', sorted(df[variable_objetivo].unique().tolist()))

df[variable_objetivo].value_counts().to_frame('registros').assign(
    porcentaje=lambda tabla: (tabla['registros'] / tabla['registros'].sum() * 100).round(2)
)

Filas duplicadas: 0
Valores de la variable objetivo: [0, 1]


,registros,porcentaje
Pago_atiempo,,
1,10252,95.25
0,511,4.75


### Conclusión de la carga

- La base se cargó correctamente con 10.763 registros y 23 columnas, y tiene todas las columnas esperadas.
- No hay filas duplicadas.
- La variable objetivo `Pago_atiempo` solo tiene los valores 0 y 1, con un desbalance marcado: 95,25 % de los clientes pagan a tiempo y 4,75 % no.
- Hay valores faltantes en 7 columnas, principalmente en `promedio_ingresos_datacredito` y `tendencia_ingresos` (cerca del 27 %).
- `fecha_prestamo` y `puntaje` deben convertirse a su tipo correcto.

El análisis detallado de cada variable se hace en `eda_analisis.ipynb`.